In [3]:
from lexguard.ingestion.pdf_loader import load_pdf
from lexguard.ingestion.chunker import build_chunks

pages = load_pdf("data/raw/sample.pdf")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print(len(chunks))
print(chunks[0])

FileNotFoundError: no such file: 'data/raw/sample.pdf'

In [ ]:
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

pages = load_text("../data/raw/sample_policy.txt")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print("Chunks:", len(chunks))
for c in chunks:
    print("-" * 40)
    print(c.clause_id)
    print(c.chunk_text)

In [4]:
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

pages = load_text("../data/raw/sample_policy.txt")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print("Chunks:", len(chunks))
for c in chunks:
    print("-" * 50)
    print("clause_id:", c.clause_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text)

Chunks: 3
--------------------------------------------------
clause_id: 1_1
section_id: 1
section_title: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.
--------------------------------------------------
clause_id: 1_3
section_id: 2
section_title: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
--------------------------------------------------
clause_id: 1_5
section_id: 3
section_title: Deletion
text: All temporary logs may be deleted after 30 days.


In [5]:
from lexguard.retrieval.bm25 import BM25Retriever

retriever = BM25Retriever(chunks)

results = retriever.query(
    "log retention period",
    top_k=2
)

for r in results:
    print("=" * 50)
    print("section:", r.section_title)
    print("text:", r.chunk_text)

section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.


In [6]:
queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\nQUERY:", q)
    results = retriever.query(q, top_k=1)
    print(results[0].chunk_text)


QUERY: log retention period
The contractor shall retain all audit logs for a minimum period of 90 days.

QUERY: how long should logs be stored
All temporary logs may be deleted after 30 days.

QUERY: financial logs retention
Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.

QUERY: log deletion policy
The contractor shall retain all audit logs for a minimum period of 90 days.


In [7]:
from lexguard.retrieval.dense import DenseRetriever

dense_retriever = DenseRetriever(chunks)

queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\n" + "=" * 60)
    print("QUERY:", q)
    results = dense_retriever.query(q, top_k=2)

    for r in results:
        print("-" * 40)
        print("section:", r.section_title)
        print("text:", r.chunk_text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4407.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUERY: log retention period
----------------------------------------
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
----------------------------------------
section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.

QUERY: how long should logs be stored
----------------------------------------
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
----------------------------------------
section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.

QUERY: financial logs retention
----------------------------------------
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.
--

In [8]:
test_queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in test_queries:
    print("\n" + "#" * 70)
    print("QUERY:", q)

    bm25_results = retriever.query(q, top_k=1)
    dense_results = dense_retriever.query(q, top_k=1)

    print("\nBM25:")
    print("section:", bm25_results[0].section_title)
    print("text:", bm25_results[0].chunk_text)

    print("\nDENSE:")
    print("section:", dense_results[0].section_title)
    print("text:", dense_results[0].chunk_text)


######################################################################
QUERY: log retention period

BM25:
section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.

DENSE:
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.

######################################################################
QUERY: how long should logs be stored

BM25:
section: Deletion
text: All temporary logs may be deleted after 30 days.

DENSE:
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.

######################################################################
QUERY: financial logs retention

BM25:
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 

In [9]:
from lexguard.retrieval.hybrid import HybridRetriever

hybrid = HybridRetriever(
    chunks,
    bm25_weight=0.5,
    dense_weight=0.5
)

queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\n" + "#" * 70)
    print("QUERY:", q)

    results = hybrid.debug_query(q, top_k=2)
    for chunk, score in results:
        print("-" * 40)
        print("score:", round(score, 4))
        print("section:", chunk.section_title)
        print("text:", chunk.chunk_text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3622.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



######################################################################
QUERY: log retention period
----------------------------------------
score: 1.0
section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.
----------------------------------------
score: 0.0985
section: Exceptions
text: Unless otherwise approved in writing by the compliance officer, logs related to financial transactions must be retained for 180 days.

######################################################################
QUERY: how long should logs be stored
----------------------------------------
score: 0.5
section: Deletion
text: All temporary logs may be deleted after 30 days.
----------------------------------------
score: 0.5
section: Data Retention
text: The contractor shall retain all audit logs for a minimum period of 90 days.

######################################################################
QUERY: financial logs retention
------------------------------